# 04 — Figures

Reads every result CSV from notebooks 01–03 and draws the figures. No
experiment runs here; nothing is recomputed except correlations already
written to CSV.

### Figure rules held to throughout

* **Small multiples, not overlays.** Each panel is one dataset x one
  sub-dimension. Nothing is pooled across datasets or sub-dimensions, so the
  figures cannot imply a composite the scoring never produced.
* **No dual-axis charts.** AUC and ECE fall and rise on different scales, so
  they get two panels, not two y-axes on one.
* **One unit.** Every axis is a fraction in `[0, 1]`; axis labels say so.
* **Fixed categorical order**, colour-blind-validated: blue, orange, aqua.
  Never more than three series in one panel — beyond that, facet.

In [ ]:
# --- Cell 1: mount Drive BEFORE importing safety_lib -------------------------
# safety_lib anchors its ROOT to the folder it lives in. If it is imported
# before Drive is mounted it either roots on /content (the scratch disk, wiped
# on disconnect) or refuses to start. Mount first, always.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# --- Cell 2: imports ---------------------------------------------------------
# Guarantees the safety_lib actually imported is the one on Drive — not a
# leftover copy in /content, which sits earlier on sys.path and wins silently.
# That is why this notebook was reading an empty /content/results.
import os
import sys
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Colab Notebooks/ai_safety_audit")
assert PROJECT.is_dir(), f"{PROJECT} not found — did cell 1 mount Drive?"

# 1. Move aside stale copies that shadow the Drive versions.
for _name in ("safety_lib.py", "dataset_specs.py"):
    _stale = Path("/content") / _name
    if _stale.exists():
        _stale.rename(_stale.with_name(_name + ".stale"))
        print(f"[path] moved aside shadowing copy: {_stale}")

# 2. Project folder FIRST on sys.path; never the working directory.
_cwd = os.getcwd()
sys.path = [p for p in sys.path if p not in ("", ".", _cwd, str(PROJECT))]
sys.path.insert(0, str(PROJECT))

# 3. Drop anything already imported from the wrong place.
for _m in ("safety_lib", "dataset_specs"):
    sys.modules.pop(_m, None)

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

import safety_lib as sl
from dataset_specs import BY_NAME

# 4. Prove it, before any figure code runs.
assert Path(sl.__file__).parent == PROJECT, f"wrong safety_lib: {sl.__file__}"
assert hasattr(sl, "paths"), (
    "old safety_lib is still on Drive — replace "
    f"{PROJECT/'safety_lib.py'} with the updated file, then restart the runtime")

pd.set_option("display.width", 220)
print(f"safety_lib {sl.VERSION} | units = {sl.UNITS}")
print(f"  imported from      {sl.__file__}")
for _k, _v in sl.paths().items():
    if _v:
        print(f"  {_k:18s} {_v}")


## Style

One place for every visual parameter. Series colours are assigned in fixed
slot order and never cycled.

In [ ]:
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]      # blue, orange, aqua (validated trio)
INK = "#0b0b0b"                                  # primary text
INK_2 = "#52514e"                                # secondary text
GRID = "#d9d8d4"
REF = "#9a9995"                                  # reference / identity lines
SURFACE = "#ffffff"

mpl.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 300,
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.titleweight": "regular",
    "axes.edgecolor": GRID, "axes.labelcolor": INK_2,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "grid.alpha": 0.7,
    "lines.linewidth": 2.0, "lines.markersize": 6,
    "legend.frameon": False, "legend.fontsize": 9,
})


def save(fig, name):
    sl.FIGURES.mkdir(parents=True, exist_ok=True)
    for ext in ("png", "pdf"):
        fig.savefig(sl.FIGURES / f"{name}.{ext}", bbox_inches="tight")
    print(f"[fig] {sl.FIGURES / name}.png  and  .pdf")
    plt.show()
    plt.close(fig)


def facet_grid(n, ncols=3, w=3.7, h=3.2):
    ncols = min(ncols, max(n, 1))
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows), squeeze=False)
    flat = axes.ravel()
    for ax in flat[n:]:
        ax.set_visible(False)
    return fig, flat


def pretty(s):
    return str(s).replace("_", " ")


def panel_title(ax, dataset, subdimension, suffix="", width=30):
    """Wrapped, fixed-size panel title — long sub-dimension names overflow otherwise."""
    import textwrap
    lines = [pretty(dataset)] + textwrap.wrap(pretty(subdimension), width)
    if suffix:
        lines.append(suffix)
    ax.set_title("\n".join(lines), color=INK, loc="left", fontsize=9.5)

## Load results

In [ ]:
def _require(name):
    """Read a results CSV, or say exactly where we looked and what is there."""
    p = sl.RESULTS / name
    if not p.exists():
        have = sorted(f.name for f in sl.RESULTS.glob("*.csv"))
        raise FileNotFoundError(
            f"{name} not found.\n"
            f"  RESULTS resolves to : {sl.RESULTS}\n"
            f"  CSVs actually there : {', '.join(have) if have else '(none)'}\n"
            f"  ROOT is anchored to : {sl.paths()['anchored_to']}\n"
            "  -> if RESULTS is not your Drive project folder, cell 2 imported the "
            "wrong safety_lib; if it IS correct, the notebook that writes this file "
            "has not been run yet.")
    return pd.read_csv(p)


# what is on disk, before anything tries to read it
_csvs = sorted(f.name for f in sl.RESULTS.glob("*.csv"))
print(f"results folder: {sl.RESULTS}")
print(f"  {len(_csvs)} CSV(s) found" + (":" if _csvs else " — nothing to plot yet"))
for _f in _csvs:
    print(f"    {_f}")
print()

inj = sl.read_results("02_injection__*.csv")
inj = inj[inj["is_target"]].copy()
cal = _require("02_calibration_summary.csv")
print(f"injection rows: {len(inj)} | calibration pairs: {len(cal)}")

try:
    pv = _require("03_predictive_validity.csv")
    joined = _require("03_pre_vs_post.csv")
    print(f"predictive-validity rows: {len(pv)} | joined runs: {len(joined)}")
except FileNotFoundError:
    pv, joined = pd.DataFrame(), pd.DataFrame()
    print("[note] notebook 03 output not found — downstream figures will be skipped")

scores = _require("01_subdimension_scores__ALL.csv")
appl = _require("01_applicability_matrix.csv")


## Figure 1 — Calibration

One panel per (dataset, sub-dimension). x is the fraction of harm actually
planted; y is what the pre-training scorer read back. Points are means over
seeds; bars are ±1 SD.

Where the score reads on the same scale as the dose (it sits at ~0 on clean
data), a grey `y = x` diagonal is drawn: the score does not have to sit on
it, it has to rise faithfully along it. Where the score has its own baseline
and range — an imbalance distance, a coverage shortfall, an out-of-fold
disagreement — the diagonal would say nothing, so the panel is marked
*own scale* and the y-axis is fitted to the data instead. What matters in
both cases is monotone response, which is what the reported `r` measures.

In [ ]:
pairs = (inj.groupby(["dataset", "subdimension", "injector"], as_index=False)
         .size().sort_values(["dataset", "subdimension"]))
fig, axes = facet_grid(len(pairs))

for ax, (_, row) in zip(axes, pairs.iterrows()):
    g = inj[(inj["dataset"] == row["dataset"]) &
            (inj["subdimension"] == row["subdimension"])]
    m = g.groupby("realized_dose")["pretraining_score"].agg(["mean", "std"]).reset_index()
    r = cal[(cal["dataset"] == row["dataset"]) &
            (cal["subdimension"] == row["subdimension"])]["pearson_r"]
    r = float(r.iloc[0]) if len(r) else np.nan

    # y = x is only a meaningful reference when the score reads on the same
    # scale as the dose — i.e. it sits at ~0 on clean data. For a score with
    # its own baseline and range (imbalance distance, coverage shortfall,
    # out-of-fold disagreement) the identity line would flatten the curve
    # against an irrelevant axis, so it is left off and the y-axis is fitted
    # to the data.
    commensurate = float(m["mean"].iloc[0]) < 0.05
    ax.errorbar(m["realized_dose"], m["mean"], yerr=m["std"].fillna(0),
                marker="o", ms=7, color=SERIES[0], capsize=3, lw=2.0, zorder=3,
                markeredgecolor=SURFACE, markeredgewidth=1.2)
    xmax = float(m["realized_dose"].max()) * 1.08 + 1e-6
    if commensurate:
        lim = max(xmax, float(m["mean"].max()) * 1.08)
        ax.plot([0, lim], [0, lim], linestyle=(0, (4, 3)), color=REF, lw=1.2, zorder=1)
        ax.set_xlim(-0.01, lim)
        ax.set_ylim(-0.01, lim)
        note = ""
    else:
        lo, hi = float(m["mean"].min()), float(m["mean"].max())
        pad = max((hi - lo) * 0.25, 0.01)
        ax.set_xlim(-0.01, xmax)
        ax.set_ylim(lo - pad, hi + pad)
        note = "own scale"
    panel_title(ax, row["dataset"], row["subdimension"],
                (f"r = {r:.3f}" if np.isfinite(r) else "")
                + (f"  ({note})" if note else ""))
    ax.set_xlabel("planted harm (fraction)")
    ax.set_ylabel("pre-training score (fraction)")

fig.suptitle("Calibration — the score tracks the harm actually planted "
             "(grey diagonal = y = x, drawn only where the scales match)",
             x=0.01, ha="left", color=INK, fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.97])
save(fig, "fig01_calibration")

## Figure 2 — Specificity: each injector moves its own sub-dimension

Only drawn when notebook 02 ran with `SCORE_ALL_APPLICABLE = True`. Each
panel is one injector on one dataset; the highlighted line is the
sub-dimension it targets, the grey lines are the other applicable
sub-dimensions on the same dataset. A clean result is one line rising and
the rest flat.

In [ ]:
allsub = sl.read_results("02_injection__*.csv")
off_target = allsub[~allsub["is_target"]]
if len(off_target):
    combos = allsub.groupby(["dataset", "injector"], as_index=False).size()
    fig, axes = facet_grid(len(combos))
    for ax, (_, row) in zip(axes, combos.iterrows()):
        g = allsub[(allsub["dataset"] == row["dataset"]) &
                   (allsub["injector"] == row["injector"])]
        shown_other = False
        for sub, gs in g.groupby("subdimension"):
            m = gs.groupby("dose")["pretraining_score"].mean()
            is_t = bool(gs["is_target"].iloc[0])
            if is_t:
                ax.plot(m.index, m.values, marker="o", ms=7, lw=2.0,
                        color=SERIES[0], zorder=3, label=f"target: {pretty(sub)}",
                        markeredgecolor=SURFACE, markeredgewidth=1.2)
            else:
                ax.plot(m.index, m.values, lw=1.2, color=REF, zorder=1,
                        label=None if shown_other else "other sub-dimensions")
                shown_other = True
        panel_title(ax, row["dataset"], f"injector: {row['injector']}")
        ax.set_xlabel("injected dose (fraction)")
        ax.set_ylabel("pre-training score (fraction)")
        ax.legend(loc="lower right", fontsize=8)
    fig.suptitle("Specificity — the injected harm moves its own sub-dimension "
                 "(grey: the others)", x=0.01, ha="left", color=INK, fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    save(fig, "fig02_specificity")
else:
    print("[skip] fig02 — notebook 02 ran with SCORE_ALL_APPLICABLE = False, "
          "so only the targeted sub-dimension was scored.")

## Figure 3 — Thresholds: where each dataset starts failing

The score curve against the tolerance for the risk level declared for that
dataset. The dashed red line is the threshold; the marked point is the
lowest dose at which the dataset fails.

In [ ]:
STATUS_BAD = "#c62828"
fig, axes = facet_grid(len(pairs))
cross_rows = []
for ax, (_, row) in zip(axes, pairs.iterrows()):
    g = inj[(inj["dataset"] == row["dataset"]) &
            (inj["subdimension"] == row["subdimension"])]
    m = g.groupby("dose")["pretraining_score"].mean()
    thr = float(g["threshold"].iloc[0])
    risk = g["risk_level"].iloc[0]

    ax.plot(m.index, m.values, marker="o", ms=7, color=SERIES[0], lw=2.0,
            markeredgecolor=SURFACE, markeredgewidth=1.2, zorder=3)
    ax.axhline(thr, color=STATUS_BAD, lw=1.6, linestyle=(0, (5, 3)), zorder=2)
    # threshold label anchored right, failure label anchored left — they would
    # otherwise collide whenever the curve crosses near the start of the axis
    ax.annotate(f"threshold {thr:.3g}  ({risk} risk)", (float(m.index[-1]), thr),
                textcoords="offset points", xytext=(0, 5), ha="right",
                color=STATUS_BAD, fontsize=8.5)

    over = m[m.values > thr]
    if len(over):
        d0 = float(over.index[0])
        ax.plot([d0], [over.iloc[0]], marker="o", ms=11, mfc="none",
                mec=STATUS_BAD, mew=2.0, zorder=4)
        ax.annotate(f"fails at dose {d0:.2f}", (d0, over.iloc[0]),
                    textcoords="offset points", xytext=(10, -14),
                    ha="left", va="top", color=STATUS_BAD, fontsize=9)
        cross_rows.append({"dataset": row["dataset"], "subdimension": row["subdimension"],
                           "risk_level": risk, "threshold": thr,
                           "first_failing_dose": d0, "units": sl.UNITS})
    else:
        cross_rows.append({"dataset": row["dataset"], "subdimension": row["subdimension"],
                           "risk_level": risk, "threshold": thr,
                           "first_failing_dose": np.nan, "units": sl.UNITS})
    panel_title(ax, row["dataset"], row["subdimension"])
    ax.set_xlabel("injected dose (fraction)")
    ax.set_ylabel("pre-training score (fraction)")

fig.suptitle("Thresholds — the dose at which each dataset fails its own "
             "sub-dimension", x=0.01, ha="left", color=INK, fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.97])
save(fig, "fig03_thresholds")
sl.write_csv(pd.DataFrame(cross_rows), "04_first_failing_dose.csv",
             n_preview=len(cross_rows))

## Figure 4 — Downstream harm vs. injected dose (tabular)

Two panels, never one with two y-axes: discrimination (AUC, falls) and
calibration error (ECE, rises) are different quantities on different scales.
Datasets are separate series in fixed colour order, capped at three.

In [ ]:
tab_files = sorted(sl.RESULTS.glob("03_posttraining_tabular__*.csv"))
if tab_files:
    tab = pd.concat([pd.read_csv(f) for f in tab_files], ignore_index=True)
    MODEL = "logreg" if "logreg" in set(tab["model"]) else sorted(tab["model"])[0]
    t = tab[tab["model"] == MODEL]

    for injector, gi in t.groupby("injector"):
        datasets = sorted(gi["dataset"].unique())[:3]
        fig, (axL, axR) = plt.subplots(1, 2, figsize=(10.4, 4.1))
        for i, ds in enumerate(datasets):
            g = gi[gi["dataset"] == ds]
            a = g.groupby("dose")["downstream_auc"].agg(["mean", "std"])
            e = g.groupby("dose")["downstream_ece"].agg(["mean", "std"])
            axL.errorbar(a.index, a["mean"], yerr=a["std"].fillna(0), marker="o", ms=7,
                         color=SERIES[i], capsize=3, lw=2.0, label=pretty(ds),
                         markeredgecolor=SURFACE, markeredgewidth=1.2)
            axR.errorbar(e.index, e["mean"], yerr=e["std"].fillna(0), marker="o", ms=7,
                         color=SERIES[i], capsize=3, lw=2.0, label=pretty(ds),
                         markeredgecolor=SURFACE, markeredgewidth=1.2)
        axL.set_title("Discrimination falls", color=INK, loc="left")
        axL.set_ylabel("held-out AUC")
        axR.set_title("Calibration error rises", color=INK, loc="left")
        axR.set_ylabel("held-out ECE (fraction)")
        for ax in (axL, axR):
            ax.set_xlabel("injected dose (fraction)")
            ax.legend(loc="best")
        fig.suptitle(f"Downstream harm — injector: {pretty(injector)}  "
                     f"(model: {MODEL}, clean held-out test set)",
                     x=0.01, ha="left", color=INK, fontsize=12)
        fig.tight_layout(rect=[0, 0, 1, 0.94])
        save(fig, f"fig04_downstream_tabular__{injector}")
else:
    print("[skip] fig04 — no results/03_posttraining_tabular__*.csv")

## Figure 5 — Predictive validity

One panel per (dataset, sub-dimension, outcome metric): the pre-training
score on x, the independently measured downstream outcome on y. Both axes
are readings — this is the forecast, not the calibration.

In [ ]:
if len(joined):
    metrics = [m for m in ("downstream_auc", "downstream_emh") if m in joined.columns]
    panels = []
    for metric in metrics:
        sub = joined.dropna(subset=[metric])
        for (ds, s, mdl), g in sub.groupby(["dataset", "targets_subdimension", "model"]):
            if len(g) >= 4:
                panels.append((ds, s, mdl, metric, g))
    if panels:
        fig, axes = facet_grid(len(panels))
        for ax, (ds, s, mdl, metric, g) in zip(axes, panels):
            ax.scatter(g["pretraining_score"], g[metric], s=44, color=SERIES[0],
                       alpha=0.85, edgecolor=SURFACE, linewidth=1.2, zorder=3)
            if g["pretraining_score"].std() > 0:
                b, a = np.polyfit(g["pretraining_score"], g[metric], 1)
                xs = np.linspace(g["pretraining_score"].min(),
                                 g["pretraining_score"].max(), 40)
                ax.plot(xs, a + b * xs, color=REF, lw=1.6,
                        linestyle=(0, (4, 3)), zorder=2)
            r = np.corrcoef(g["pretraining_score"], g[metric])[0, 1]
            panel_title(ax, f"{ds} · {mdl}",
                        f"{pretty(s)} -> {pretty(metric)}", f"r = {r:.3f}")
            ax.set_xlabel("pre-training score (fraction)")
            ax.set_ylabel(pretty(metric))
            # narrow score ranges otherwise produce colliding 6-digit ticks
            ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(5))
        fig.suptitle("Predictive validity — the pre-training score forecasts the "
                     "independently measured outcome", x=0.01, ha="left",
                     color=INK, fontsize=12)
        fig.tight_layout(rect=[0, 0, 1, 0.97])
        save(fig, "fig05_predictive_validity")
else:
    print("[skip] fig05 — no results/03_pre_vs_post.csv")

## Figure 6 — Coverage map

Which sub-dimension was scored for which dataset, and what happened to the
rest. State is carried by the cell text, not by colour alone.

In [ ]:
state = appl.assign(
    state=np.where(~appl["enabled"], "off",
           np.where(appl["applicable"], "scored", "N/A")))
piv = state.pivot_table(index="subdimension", columns="dataset",
                        values="state", aggfunc="first")
order = [s.id for s in sl.CATALOG if s.id in piv.index]
piv = piv.loc[order]
fill = {"scored": "#cde2fb", "N/A": "#f2f1ee", "off": "#fbe3d8"}
label = {"scored": "scored", "N/A": "N/A", "off": "off"}

fig, ax = plt.subplots(figsize=(1.9 * len(piv.columns) + 3.4, 0.46 * len(piv) + 1.6))
ax.set_xlim(0, len(piv.columns))
ax.set_ylim(0, len(piv))
ax.grid(False)
for sp in ax.spines.values():
    sp.set_visible(False)
for i, sub in enumerate(piv.index):
    y = len(piv) - 1 - i
    for j, ds in enumerate(piv.columns):
        v = piv.iloc[i, j]
        ax.add_patch(plt.Rectangle((j + 0.03, y + 0.08), 0.94, 0.84,
                                   facecolor=fill.get(v, "#ffffff"),
                                   edgecolor=SURFACE, linewidth=2))
        ax.text(j + 0.5, y + 0.5, label.get(v, "?"), ha="center", va="center",
                fontsize=9, color=INK if v == "scored" else INK_2)
ax.set_xticks(np.arange(len(piv.columns)) + 0.5)
ax.set_xticklabels([pretty(c) for c in piv.columns], fontsize=9, color=INK_2)
ax.set_yticks(np.arange(len(piv)) + 0.5)
ax.set_yticklabels([pretty(s) for s in reversed(list(piv.index))],
                   fontsize=9, color=INK_2)
ax.tick_params(length=0)
dims = {s.id: s.dimension for s in sl.CATALOG}
ax.set_title("Coverage — every sub-dimension accounted for, per dataset\n"
             "(N/A reasons are in results/01_applicability_matrix.csv)",
             loc="left", color=INK, fontsize=12)
fig.tight_layout()
save(fig, "fig06_coverage_map")

## Summary table

One row per (dataset, sub-dimension): the clean score, the calibration
against planted harm, and — where notebook 03 ran — the predictive validity.
Deliberately long-format: there is no column that averages sub-dimensions.

In [ ]:
clean = (scores[scores["applicable"] == True]
         [["dataset", "dimension", "subdimension", "score", "threshold",
           "exceeds_threshold"]]
         .rename(columns={"score": "clean_score"}))
summary = clean.merge(
    cal[["dataset", "subdimension", "pearson_r", "ci_low", "ci_high", "r2"]]
       .rename(columns={"pearson_r": "calibration_r", "ci_low": "calibration_ci_low",
                        "ci_high": "calibration_ci_high", "r2": "calibration_r2"}),
    on=["dataset", "subdimension"], how="left")
if len(pv):
    best = (pv.sort_values("pearson_r", key=lambda s: s.abs(), ascending=False)
              .groupby(["dataset", "subdimension"], as_index=False).first()
              [["dataset", "subdimension", "outcome_metric", "model",
                "pearson_r", "confirmed"]]
              .rename(columns={"pearson_r": "predictive_r",
                               "outcome_metric": "predictive_outcome",
                               "model": "predictive_model"}))
    summary = summary.merge(best, on=["dataset", "subdimension"], how="left")
summary["units"] = sl.UNITS
print(summary.round(4).to_string(index=False))
sl.write_csv(summary, "04_summary_by_dataset_subdimension.csv", n_preview=len(summary))

In [ ]:
print(f"\nfigures written to {sl.FIGURES}")
for f in sorted(sl.FIGURES.glob("*.png")):
    print("  ", f.name)